## Import

In [189]:
from bertopic import BERTopic
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import csv
from evaluation import evaluate_model
from bertopic.vectorizers import ClassTfidfTransformer

df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])
classes = list(df["gen"])

In [2]:
# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 32/32 [00:15<00:00,  2.05it/s]


In [249]:
#from sklearn.cluster import KMeans

run_name = "BERTopic_outlier_reduction_prob_005_nr_topics_15"

# Hyperparameters

n_neighbors = 30 # 15 -> BEST 30
n_components = 5 # 5 -> BEST 5
random_state = [0, 1, 37, 42, 73]
min_dist = 0.0
min_cluster_size = 10 # 10 -> BEST 10
min_df = 2 # 2 -> BEST 2 BUT 1 GOOD
ngram_range = (1, 2) # (1, 2) -> BEST (1, 2)
top_n_words = 10 # 10 -> BEST 10
#nr_topics = 15 # auto

# Data saving

data = []

# Setup different models

cluster_model = HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
#luster_model = KMeans(n_clusters=15)
vectorizer_model = CountVectorizer(stop_words="english", min_df=min_df, ngram_range=ngram_range)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True) # enable by default --> BEST DEFAULT

# Train

for seed in random_state:
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=0.0, metric='cosine', random_state=seed)

    topic_model = BERTopic(

        # Pipeline models
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        #ctfidf_model=ctfidf_model,

        # Hyperparameters
        top_n_words=top_n_words,
        n_gram_range=ngram_range,
        min_topic_size="auto", #use HDBSCAN
        verbose=True,
        #nr_topics = nr_topics, # base is auto

        # General parameters
        calculate_probabilities=True,
        language="english"
    )

    topics, probs = topic_model.fit_transform(docs, embeddings)
    
    # Outlier reduction
    new_topics = topic_model.reduce_outliers(docs, topics, probabilities=probs, strategy="probabilities", threshold=0)
    # new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings")
    topic_model.update_topics(docs,
                              topics=new_topics,
                              vectorizer_model=vectorizer_model,
                              ctfidf_model=ctfidf_model
                              )

    # Evaluate model

    scores = evaluate_model(topic_model, docs, topk=topic_model.top_n_words)
    
    data.append([seed, scores[0], scores[1]])

df = pd.DataFrame(data, columns=["seed", "Diversity", "Coherence"])
df

2025-01-31 14:45:28,376 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-01-31 14:45:30,980 - BERTopic - Dimensionality - Completed ✓
2025-01-31 14:45:30,981 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-01-31 14:45:31,034 - BERTopic - Cluster - Completed ✓
2025-01-31 14:45:31,037 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-01-31 14:45:31,125 - BERTopic - Representation - Completed ✓
2025-01-31 14:45:31,235 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
100%|██████████| 1/1 [00:00<00:00,  2.54it/s]
2025-01-31 14:45:33,924 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-01-31 1

,seed,Diversity,Coherence
0,0,0.806250,-0.113407
1,1,0.823529,-0.076110
2,37,0.805882,-0.058474
3,42,0.817647,-0.058824
4,73,0.830000,-0.037813


Save model

In [165]:
embedding_model = "all-MiniLM-L6-v2"
topic_model.save(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", run_name), serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

Save data

In [166]:
param_str = str({
    "n_neighbors": n_neighbors,
    "n_components": n_components,
    "min_dist": min_dist,
    "random_state": random_state,
    "min_cluster_size": min_cluster_size,
    "min_df": min_df,
    "ngram_range": ngram_range,
    "top_n_words": top_n_words
})


data_run = [[run_name, param_str, df["Diversity"].mean(), df["Coherence"].mean()]]

df_run = pd.DataFrame(data_run, columns=["run_name", "params", "diversity", "coherence"])

df_run.to_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", "results.csv"), mode="a", header=False, index=False)

Show results

In [172]:
df_results = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", "results.csv"))
df_results

,run_name,params,diversity,coherence
0,base,"{'n_neighbors': 15, 'n_components': 5, 'min_di...",0.518766,0.018410
1,UMAP_n_neighbors_20,"{'n_neighbors': 20, 'n_components': 5, 'min_di...",0.508158,0.015145
2,UMAP_n_neighbors_25,"{'n_neighbors': 25, 'n_components': 5, 'min_di...",0.539043,0.011082
3,UMAP_n_neighbors_30,"{'n_neighbors': 30, 'n_components': 5, 'min_di...",0.543632,0.024256
4,UMAP_n_neighbors_35,"{'n_neighbors': 35, 'n_components': 5, 'min_di...",0.533987,0.014548
5,UMAP_n_neighbors_40,"{'n_neighbors': 40, 'n_components': 5, 'min_di...",0.529626,0.021350
6,UMAP_n_neighbors_45,"{'n_neighbors': 45, 'n_components': 5, 'min_di...",0.537632,0.020211
7,UMAP_n_neighbors_50,"{'n_neighbors': 50, 'n_components': 5, 'min_di...",0.520158,0.022319
8,UMAP_n_neighbors_55,"{'n_neighbors': 55, 'n_components': 5, 'min_di...",0.528634,0.025503
9,UMAP_n_neighbors_60,"{'n_neighbors': 60, 'n_components': 5, 'min_di...",0.532111,0.023156


In [250]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,138,0_mushrooms_pair_number mushrooms_pattern,"[mushrooms, pair, number mushrooms, pattern, m...",[There's a pattern when it comes to the gnomes...
1,1,114,1_basket_yellow basket_red basket_baskets,"[basket, yellow basket, red basket, baskets, r...","[The blue, yellow and green gnomes show the ye..."
2,2,82,2_baskets_basket_basket colour_red basket,"[baskets, basket, basket colour, red basket, c...",[On the screen you will be presented with two ...
3,3,90,3_mushroom_colours mushrooms_colours_mushrooms,"[mushroom, colours mushrooms, colours, mushroo...",[You should try and test multiple colors to se...
4,4,66,4_scores_points_went_pink blue,"[scores, points, went, pink blue, pick, colors...",[there are similar colour patterns that come o...
5,5,54,5_points_earthy_group_points gnome,"[points, earthy, group, points gnome, gnomes p...",[Gnomes will provide points varying from 0-9 b...
6,6,34,6_hats_tall_hat_tall hats,"[hats, tall, hat, tall hats, hat tall, taller,...","[Do your best, seems pretty random and difficu..."
7,7,31,7_keys_fingers_hand_breaks,"[keys, fingers, hand, breaks, hands, focused, ...","[As each gnome appears, tap S for the left gno..."
8,8,26,8_forest_gnomes forest_forests_switch forest,"[forest, gnomes forest, forests, switch forest...","[There are two forests. Green, orange, yellow,..."
9,9,28,9_knomes_earners_knome_purple blue,"[knomes, earners, knome, purple blue, brown us...","[try to choose blue, green and yellow. Avoid b..."


In [251]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [252]:
topic_model.visualize_documents(docs)

## DYNAMIC

In [253]:
topics_over_time = topic_model.topics_over_time(docs, classes)

topic_model.visualize_topics_over_time(topics_over_time)

0it [00:00, ?it/s]

10it [00:00, 14.41it/s]


In [260]:
a = topic_model.probabilities_
b = topic_model.get_document_info(docs)